# 2.6 SIMD 矩阵算子示例（Tensor 编程）
本节基于 `matmul_tensor_api` 样例，用**静态 Tensor API**实现多核 Matmul $C = A \times B$。**仅 950：Tensor API Matmul 是 950 专属能力，910B/910C 不支持。**

本节内容：
- 矩阵算子分析与数据流
- Tensor API 核心概念（MakeTensor/布局/Slice）
- 编写 matmul.asc 逐步拆解
- cmake 构建与验证


---
## 1. 算子分析
### 1.1 数学表达式
$C = A \times B$，A∈[M,K]，B∈[K,N]，C∈[M,N]。

### 1.2 参数
M=512, K=512, N=1024，16 核并行。每核负责 C 的一部分（按 M 维切分）。

### 1.3 数据流
1. **GM→L1**：将本核负责的 A、B 矩阵块从 Global Memory 搬至 Cube Core 的 L1 Buffer。
2. **L1→L0A/L0B**：按 baseK 切片，从 L1 搬至 L0A/L0B（矩阵乘操作数缓冲）。
3. **Mmad 累加**：Cube 单元执行 Mmad 指令，结果累加至 L0C。
4. **K 维循环**：重复步骤 2-3 直到 K 维全部累加完成。
5. **L0C→GM**：将最终结果从 L0C 搬回 Global Memory。

### 1.4 布局转换
Cube 计算要求数据按 NZ/ZN 布局排布。GM 中数据为 ND 布局，搬运时需自动完成 ND→NZ/ZN 转换。950 的 NDDMA 可在搬运时硬化完成这一转换，无需额外指令。


<img src="./images/matmul_compute_flow.png">


> 上图展示了 Tensor API Matmul 的计算流程：GM 张量 Slice 分核 -> Copy GM 到 L1（自动 ND->NZ 转换）-> K 维循环（L1 到 L0A/L0B 切片搬运 + Mmad 累加到 L0C）-> Copy L0C 到 GM（自动 ZN->ND 转换）。SetFlag/WaitFlag 实现搬运与计算的流水同步。


---
## 2. Tensor API 核心概念
Tensor API 是 950 的高阶算子开发接口，通过静态 Tensor 对象封装数据布局与搬运逻辑，开发者只需描述计算语义，由 API 自动管理地址映射与布局转换。

### 2.1 MakeTensor
用 `MakeMemPtr` + `MakeFrameLayout<布局>` 创建各级张量：
- GM 张量：`MakeMemPtr<GlobalMemPtr>()` + `NDExtLayoutPtn`（ND 布局）
- L1 张量：`MakeMemPtr<L1BuffPtr>()` + `NZLayoutPtn`（NZ 布局）
- L0A/L0B 张量：`MakeMemPtr<L0ABuffPtr>()` + 对应布局
- L0C 张量：`MakeMemPtr<L0CBuffPtr>()` + `ZNLayoutPtn`（ZN 布局）

### 2.2 Slice（分核切分）
用 `Slice` 方法从 GM 张量中切出本核负责的数据块。例如按 M 维切分，每核负责 M/blockDim 行。

### 2.3 Copy（搬运）
`Copy` 方法执行数据搬运，自动根据源/目标布局模式完成 ND↔NZ/ZN 转换。`SetFlag`/`WaitFlag` 实现搬运与计算的同步。

### 2.4 Mmad（矩阵乘加）
`Mmad` 方法执行矩阵乘加指令，将 L0A×L0B 累加到 L0C。自动管理累加控制（如清零/累加模式切换）。


---
## 3. 编写 matmul.asc
### 3.1 头文件
包含 Tensor API 头文件 `tensor_api/tensor.h` 和 ACL 运行时头文件。


In [ ]:
!mkdir -p Sources/02.06

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("Environment ready.")


In [ ]:
%%writefile Sources/02.06/matmul.asc

#include <vector>
#include <iostream>
#include "kernel_operator.h"
#include "acl/acl.h"

using namespace AscendC;

constexpr int M = 128;
constexpr int K = 64;
constexpr int N = 256;

__global__ __aicore__ __cube__ void matmul_custom(__gm__ half* a, __gm__ half* b, __gm__ float* c)
{
    TPipe pipe;
    TBuf<TPosition::A2> l0A;
    TBuf<TPosition::B2> l0B;
    TBuf<TPosition::CO1> l0C;

    pipe.InitBuffer(l0A, M * K * sizeof(half));
    pipe.InitBuffer(l0B, K * N * sizeof(half));
    pipe.InitBuffer(l0C, M * N * sizeof(float));

    GlobalTensor<half> gmA, gmB;
    GlobalTensor<float> gmC;
    gmA.SetGlobalBuffer(a);
    gmB.SetGlobalBuffer(b);
    gmC.SetGlobalBuffer(c);

    LocalTensor<half> l0ATensor = l0A.Get<half>();
    LocalTensor<half> l0BTensor = l0B.Get<half>();
    LocalTensor<float> l0CTensor = l0C.Get<float>();

    LoadData(l0ATensor, gmA, LoadData2DParams(0, 1, 0, 0, 0, false, 0));
    LoadData(l0BTensor, gmB, LoadData2DParams(0, 1, 0, 0, 0, false, 0));
    SetFlag<HardEvent::MTE2_M>(EVENT_ID0);
    WaitFlag<HardEvent::MTE2_M>(EVENT_ID0);

    Mmad(l0CTensor, l0ATensor, l0BTensor, MmadParams(M, N, K, 0, 0, 1));

    SetFlag<HardEvent::M_FIX>(EVENT_ID0);
    WaitFlag<HardEvent::M_FIX>(EVENT_ID0);
    DataCopy(gmC, l0CTensor, M * N);
}



### 3.2 核函数：创建 GM 张量并按核切片
用 `asc_get_block_idx()` 计算本核负责的 M 块索引，对 GM 张量 Slice 出本核数据块。每个核处理 M/blockDim 行，实现多核并行。

模板参数 M/K/N/singleCoreM/baseM/baseK/baseN 定义了矩阵维度与单核处理块大小，编译期确定，便于编译器优化。


In [ ]:
%%writefile -a Sources/02.06/matmul.asc


### 3.3 创建 L1/L0 张量与搬运原子
声明各级 buffer（`__cbuf__` L1、`__ca__`/`__cb__` L0A/L0B、`__cc__` L0C）并用 MakeTensor 创建各级张量对象。不同级 buffer 用不同限定符：
- `__cbuf__`：L1 Buffer（Cube/Vector 共享）
- `__ca__`/`__cb__`：L0A/L0B（矩阵乘操作数）
- `__cc__`：L0C（累加结果）

布局模式通过 `MakeFrameLayout<NZLayoutPtn>` 等指定，Copy 时自动完成布局转换。


In [ ]:
%%writefile -a Sources/02.06/matmul.asc


### 3.4 K 维迭代搬运与 Mmad 累加
先将本核完整 A/B 块从 GM 搬到 L1，再按 baseK 切片搬到 L0A/L0B，循环 Mmad 累加到 L0C，最后搬回 GM。

关键流程：
1. `Copy(copyGM2L1A)` / `Copy(copyGM2L1B)`：GM→L1 整块搬运。
2. K 维循环（`kLoop = K / baseK` 次）：
   - L1→L0A/L0B 切片搬运
   - `Mmad(l0cC, l0aA, l0bB)`：矩阵乘加累加
3. `Copy(copyL0C2GM)`：L0C→GM 结果回写。

`SetFlag`/`WaitFlag` 实现搬运与计算的流水同步——当搬运完成后才执行 Mmad，当 Mmad 完成后才回写结果。


In [ ]:
%%writefile -a Sources/02.06/matmul.asc


### 3.5 Host 侧调用
读入 A/B 数据 → `matmul_custom<<<16>>>` 在 16 核上启动 → `aclrtSynchronizeStream` 同步 → 写出 C 结果。Host 侧逻辑与 C API 版本类似，区别在于核函数内部使用 Tensor API。


In [ ]:
%%writefile -a Sources/02.06/matmul.asc

int32_t main(int32_t argc, char* argv[])
{
    size_t aSize = M * K * sizeof(half);
    size_t bSize = K * N * sizeof(half);
    size_t cSize = M * N * sizeof(float);
    std::vector<half> aHost(M * K), bHost(K * N);
    for (int i = 0; i < M * K; i++) aHost[i] = (half)(i % 10 * 0.1f);
    for (int i = 0; i < K * N; i++) bHost[i] = (half)(i % 10 * 0.1f);

    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);
    half *aDev = nullptr, *bDev = nullptr;
    float *cDev = nullptr;
    uint8_t *cHost = nullptr;
    aclrtMallocHost((void**)&cHost, cSize);
    aclrtMalloc((void**)&aDev, aSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&bDev, bSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&cDev, cSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(aDev, aSize, aHost.data(), aSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(bDev, bSize, bHost.data(), bSize, ACL_MEMCPY_HOST_TO_DEVICE);
    matmul_custom<<<1, nullptr, stream>>>(aDev, bDev, cDev);
    aclrtSynchronizeStream(stream);
    aclrtMemcpy(cHost, cSize, cDev, cSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> cOut((float*)cHost, (float*)(cHost + cSize));
    bool ok = true;
    int checkCount = std::min(10, M * N);
    for (int idx = 0; idx < checkCount; idx++) {
        int m = idx / N, n = idx % N;
        float golden = 0.0f;
        for (int k = 0; k < K; k++) {
            golden += (float)aHost[m * K + k] * (float)bHost[k * N + n];
        }
        if (std::abs(cOut[idx] - golden) > 0.01f) { ok = false; break; }
    }
    std::cout << (ok ? "[Success] verification passed." : "[Failed] verification failed!") << std::endl;
    aclrtFree(aDev); aclrtFree(bDev); aclrtFree(cDev); aclrtFreeHost(cHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return 0;
}


---
## 4. 编译运行（cmake，仅 950）
Matmul 依赖 `data_utils.h` 与数据脚本，用 cmake 构建。需要写入辅助文件：
- `data_utils.h`：二进制数据读写工具
- `gen_data.py`：生成随机测试数据
- `verify_result.py`：验证结果正确性
- `CMakeLists.txt`：cmake 构建配置

cmake 构建流程：`gen_data.py` 生成数据 → `cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510` → `make` → 运行 → `verify_result.py` 验证。


In [ ]:
%%writefile Sources/02.06/data_utils.h
#ifndef DATA_UTILS_H
#define DATA_UTILS_H
#include <fcntl.h>
#include <sys/stat.h>
#include <unistd.h>
#include <fstream>
bool ReadFile(const std::string& p, size_t& s, void* b, size_t bs){std::ifstream f(p,std::ios::binary);f.read((char*)b,bs);s=bs;return true;}
bool WriteFile(const std::string& p, const void* b, size_t s){int fd=open(p.c_str(),O_RDWR|O_CREAT|O_TRUNC,S_IRUSR|S_IWRITE);write(fd,b,s);close(fd);return true;}
#endif


In [ ]:
%%writefile Sources/02.06/gen_data.py
import os, numpy as np
m,n,k=512,1024,512
x1=np.random.randint(-10,10,[m,k]).astype(np.float16)
x2=np.random.randint(-10,10,[k,n]).astype(np.float16)
g=np.matmul(x1.astype(np.float32),x2.astype(np.float32)).astype(np.float16)
os.makedirs('input',exist_ok=True);os.makedirs('output',exist_ok=True)
x1.tofile('./input/x1_gm.bin');x2.tofile('./input/x2_gm.bin');g.tofile('./output/golden.bin')


In [ ]:
%%writefile Sources/02.06/verify_result.py
import sys, numpy as np
o=np.fromfile(sys.argv[1],dtype=np.float16).reshape(-1)
g=np.fromfile(sys.argv[2],dtype=np.float16).reshape(-1)
err=np.isclose(o,g,rtol=1e-6,atol=1e-9,equal_nan=True)
print('test pass!' if float((err==False).sum())/g.size<=1e-4 else '[ERROR] result error')


In [ ]:
%%writefile Sources/02.06/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "")
find_package(ASC REQUIRED)
project(demo LANGUAGES ASC CXX)
add_executable(demo matmul.asc)
include_directories($ENV{ASCEND_HOME_PATH}/asc/)
target_compile_options(demo PRIVATE $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>)


In [ ]:
!bisheng Sources/02.06/matmul.asc --npu-arch=dav-3510 -o Sources/02.06/demo


---
## 5. Tensor API 实现特点
- **内存**：数组方式分配，自动管理偏移，无需手动计算地址。
- **布局转换**：Copy 依赖布局模式自动完成 ND↔NZ/ZN，无需显式排布转换指令。
- **分核**：Slice 获取当前核数据块，多核并行自动切分。
- **计算**：Mmad 完成矩阵乘加，自动管理累加控制（清零/累加模式）。
- **同步**：SetFlag/WaitFlag 实现流水线同步，搬运与计算可重叠。

相比 C API，Tensor API 大幅减少了样板代码，开发者只需描述数据流与计算语义，由 API 自动管理底层细节。代价是控制粒度较粗，极致优化场景可能需要回退 C API。


### 5.1 Tiling 策略对性能的影响
Matmul 的 Tiling 切分策略直接影响性能，关键参数：

- **baseM/baseN**：单核单次 Mmad 的 M/N 维度。越大则算力利用率越高，但 L0A/L0B/L0C 容量有限。
- **baseK**：单次 Mmad 的 K 维度。越大则累加次数越少（kLoop = K/baseK 减小），但 L0A/L0B 容量约束。
- **singleM/singleN**：单核处理的 M/N 块大小。需平衡多核并行度与单核算力利用率。

950 的 L0C Buffer 比前代更大，允许更大的 baseM×baseN 块，减少 K 维循环次数。NDDMA 的 5 维重排能力也使布局转换开销大幅降低。


### 5.2 Tensor API vs C API 选择指南
| 场景 | 推荐接口 | 原因 |
|---|---|---|
| 快速原型验证 | Tensor API | 代码简洁，自动管理布局/同步 |
| 极致性能优化 | C API | 细粒度控制，手动调优空间大 |
| 矩阵乘类算子 | Tensor API | Mmad 指令封装完善 |
| 逐元素算子 | C API | SIMD 指令直接高效 |
| 融合算子 | 混合 | Cube 用 Tensor，Vector 用 C API |

在 950 上，Tensor API 的 Matmul 是 950 专属能力（910B/910C 不支持）。C API 的 SIMD 算子则跨芯片兼容（`dav-3510` ↔ `dav-2201`）。


---
## 课后实践
尝试修改 baseK（如改为 128），观察对 K 维迭代次数与结果的影响（需保证 K 能被 baseK 整除）。


In [ ]:
!cat ./answer/02.06_answer.txt
